In [ ]:
!pip install transformers datasets torch torchaudio librosa captum

In [ ]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio
import torch
import librosa

import torch.nn.functional as F
from captum.attr import IntegratedGradients

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
TARGET_IDX = 525

In [ ]:
model_id = "VasilisAsim/data2vec-finetuned-IEMOCAP-categorical"
model_ft = AutoModelForAudioClassification.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model_ft.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft.to(device)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

def merge_and_filter(batch):
    emo = batch["major_emotion"]
    if emo == "excited":
        emo = "happy"
    batch["emotion_4class"] = emo
    return batch

dataset = dataset.map(merge_and_filter)
dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])

def add_speaker(batch):
    parts = batch["file"].split("_")
    session = parts[0][:5]
    speaker_gender = parts[-1][0]
    batch["speaker_id"] = f"{session}_{speaker_gender}"
    return batch

dataset = dataset.map(add_speaker)

labels = sorted(set(dataset["emotion_4class"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}
print("Κλάσεις:", labels)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx),
})

In [ ]:
MAX_SAMPLES = 6 * 16_000

def is_valid_audio(batch):
    audio = batch["audio"]["array"][:MAX_SAMPLES]
    if len(audio) == 0:
        return False
    return np.std(audio) >= 1e-4

print("Πριν:", len(dataset["train"]), len(dataset["test"]))
dataset["train"] = dataset["train"].filter(is_valid_audio, load_from_cache_file=False)
dataset["test"] = dataset["test"].filter(is_valid_audio, load_from_cache_file=False)
print("Μετά:", len(dataset["train"]), len(dataset["test"]))

In [ ]:
#Wrapper function για την μετατροπή της εξόδου σε tensors
def forward_func(input_values):
    outputs = model_ft(input_values)
    return F.softmax(outputs.logits, dim=-1)

In [ ]:
ig = IntegratedGradients(forward_func)

In [ ]:
def explain_sample(idx, split="test"):
    sample = dataset[split][idx]
    speech = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if sr != feature_extractor.sampling_rate:
        speech = librosa.resample(speech, orig_sr=sr, target_sr=feature_extractor.sampling_rate)

    inputs = feature_extractor(speech, sampling_rate=feature_extractor.sampling_rate, max_length=6*16000, truncation=True, return_tensors="pt")
    input_values = inputs["input_values"].to(device).requires_grad_()

    true_label = sample["emotion_4class"]

    with torch.no_grad():
        pred_class = forward_func(input_values).argmax(dim=-1).item()

    baseline = torch.zeros_like(input_values)
    attributions, delta = ig.attribute(
    input_values, baselines=baseline, target=pred_class,
    n_steps=50, internal_batch_size=2,
    return_convergence_delta=True
)

    return speech, attributions.squeeze().detach().cpu().numpy(), pred_class, true_label, delta.item()

In [ ]:
speech, attr, pred_class, true_label, delta = explain_sample(TARGET_IDX, split="test")

attr_abs = np.abs(attr)
window = 400
attr_smooth = np.convolve(attr_abs, np.ones(window)/window, mode='same')
max_len = 6 * 16000
speech = speech[:max_len]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(speech, color='gray')
axes[0].set_title(f"Κυματομορφή — True label: {true_label}")

axes[1].plot(attr_smooth, color='red')
axes[1].set_title(f"Integrated Gradients — Predicted: {model_ft.config.id2label[pred_class]}")
plt.tight_layout()
plt.show()